In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
import matplotlib.pyplot as plt
import datetime, time
import os
from dotenv import load_dotenv

import matplotlib.pyplot as plt

import re
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)


In [2]:

def establish_db_connection(server, database, username, password, driver):
    connection_string = (
        f"mssql+pyodbc://{username}:{password}@{server}/{database}"
        f"?driver={driver.replace(' ', '+')}"
    )
    engine = create_engine(connection_string)

    try:
        with engine.connect() as conn:
            result = conn.execute(text("SELECT @@VERSION"))
            for row in result:
                print("Connected successfully. SQL Server version:")
                print(row[0])
            return engine
    except Exception as e:
        print("Connection failed:")
        print(e)
        return None

# Adjust to include error handling for the db connection method



In [3]:
def load_env():
    load_dotenv(dotenv_path="creds\\.env")


def SERVER_conn(input_site):

    load_env()

    # DB server
    site_server = os.getenv(input_site)
    
    
    paramz = {
        "site": os.getenv('site_server'),
        "userName": os.getenv('USER_NAME'),
        "Password": os.getenv('PASSWORD_dev-test'),
        "Driver": os.getenv("ODBC_DRIVER")
    }

    db = os.getenv(input_site)

    server_conn = establish_db_connection(
        paramz["site"],
        db, 
        paramz["userName"],     
        paramz["Password"],
        paramz["Driver"])
        
    return server_conn


def db_request(query, server_conn_str):
    if server_conn_str is None:
        raise Exception("Database connection failed. Please check your credentials and connection settings.")

    # start_time = time.time()
    df = pd.read_sql(query, server_conn_str)

    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"db_request must return a DataFrame, got {type(df)}")

    # end_time = time.time()
    # print(f"Query executed in {end_time - start_time:.2f} seconds")
    return df


In [4]:

# RO_all = "SELECT *  from Ops_tblRepairOrder where fldLastUpdated > '2020-01-1' AND fldStatus = 3 AND fldDivision IN (1)"
# query_all_requests = "SELECT *  from Ops_tblRequests where fldLastUpdated > '2020-01-1' AND fldAddWorkStatus IN (100, 300, 400)" 
# query_all_LabourLine = "SELECT *  from Ops_tblLabourLine where fldLastUpdated > '2020-01-1'"
# query_all_PartsLine = "SELECT *  from Ops_tblPartsLine where fldLastUpdated > '2020-01-1'"

# # More queries
# 
# 
# 

# get all F150 closed RO with relevant requests
RO_all = "SELECT RO.fldId, RO.fldContactRef, RO.fldVehicleRef, RO.fldDateOpened, RO.fldDateClosed FROM Ops_tblRepairOrder RO WITH(NOLOCK) INNER JOIN Ops_tblRequests REQ WITH(NOLOCK) ON RO.fldId = REQ.fldWorkItemRef INNER JOIN Veh_tblVehicle VEH WITH(NOLOCK) ON VEH.fldId = RO.fldVehicleRef INNER JOIN Veh_tblTrim TR WITH(NOLOCK) ON TR.fldId = VEH.fldTrimRef INNER JOIN Veh_tblModel MOD WITH(NOLOCK) ON MOD.fldId = TR.fldModelRef WHERE 1=1 AND RO.fldStatus = 3 AND RO.fldDivision IN (1) AND MOD.fldName in ('F150', 'F-150') AND REQ.fldAddWorkStatus IN (100, 300, 400)"

query_all_requests = "SELECT Req.fldId, Req.fldWorkItemRef, Req.fldSequence, Req.fldDescription, Req.fldRequestCodeRef, Req.fldRequestCode, Req.fldRequestedTime, Req.fldOrderNumber, Req.fldLastUpdated FROM Ops_tblRepairOrder RO WITH(NOLOCK) INNER JOIN Ops_tblRequests REQ WITH(NOLOCK) ON RO.fldId = REQ.fldWorkItemRef INNER JOIN Veh_tblVehicle VEH WITH(NOLOCK) ON VEH.fldId = RO.fldVehicleRef INNER JOIN Veh_tblTrim TR WITH(NOLOCK) ON TR.fldId = VEH.fldTrimRef INNER JOIN Veh_tblModel MOD WITH(NOLOCK) ON MOD.fldId = TR.fldModelRef  WHERE 1=1 AND RO.fldStatus = 3 AND RO.fldDivision IN (1) AND MOD.fldName in ('F150', 'F-150') AND REQ.fldAddWorkStatus IN (100, 300, 400)" 
query_all_PartsLine = "SELECT PL.fldID, PL.fldRequestRef, PL.fldSequence, PL.fldPartNumber, PL.fldPartDesc, PL.fldRequested, PL.fldShipped, PL.fldOrderType, PL.fldDateAdded FROM Ops_tblRepairOrder RO WITH(NOLOCK) INNER JOIN Ops_tblRequests REQ WITH(NOLOCK) ON RO.fldId = REQ.fldWorkItemRef INNER JOIN Veh_tblVehicle VEH WITH(NOLOCK) ON VEH.fldId = RO.fldVehicleRef INNER JOIN Veh_tblTrim TR WITH(NOLOCK) ON TR.fldId = VEH.fldTrimRef INNER JOIN Veh_tblModel MOD WITH(NOLOCK) ON MOD.fldId = TR.fldModelRef  INNER JOIN Ops_tblPartsLine PL WITH(NOLOCK) ON PL.fldRequestRef = REQ.fldId WHERE 1=1 AND RO.fldStatus = 3 AND RO.fldDivision IN (1) AND MOD.fldName in ('F150', 'F-150') AND REQ.fldAddWorkStatus IN (100, 300, 400)"

query_all_LabourLine = f"SELECT LL.fldID, LL.fldRequestRef, LL.fldOpCodeRef, LL.fldActualHours, LL.fldSoldHours, LL.fldDescription, LL.fldAddedDate FROM Ops_tblRepairOrder RO WITH(NOLOCK) INNER JOIN Ops_tblRequests REQ WITH(NOLOCK) ON RO.fldId = REQ.fldWorkItemRef INNER JOIN Veh_tblVehicle VEH WITH(NOLOCK) ON VEH.fldId = RO.fldVehicleRef INNER JOIN Veh_tblTrim TR WITH(NOLOCK) ON TR.fldId = VEH.fldTrimRef INNER JOIN Veh_tblModel MOD WITH(NOLOCK) ON MOD.fldId = TR.fldModelRef INNER JOIN Ops_tblLabourLine LL WITH(NOLOCK) ON LL.fldRequestRef = REQ.fldId WHERE 1=1 AND RO.fldStatus = 3 AND RO.fldDivision IN (1) AND MOD.fldName in ('F150', 'F-150') AND REQ.fldAddWorkStatus IN (100, 300, 400)"
 

In [5]:
# search for op_codes_based_on_key_words

# select * from 
# Ops_tblOpCode2
# where fldDescription like ('%Water Pump%')

In [6]:

def pull_data_by_server(server_conn_str):
    # pull data for 
    RO_tbl = db_request(RO_all, server_conn_str)
    request_tbl = db_request(query_all_requests, server_conn_str)
    labourline_tbl = db_request(query_all_LabourLine, server_conn_str)
    partslines_tbl = db_request(query_all_PartsLine, server_conn_str)

    return RO_tbl, request_tbl, labourline_tbl, partslines_tbl

def pull_data_by_server_with_args(server_conn_str, queries, modelName):
    # pull data for 

    # RO_tbl = server_conn_str.execute(queries["RO_tbl"], {"model": modelName}).fetchall()
    # request_tbl = server_conn_str.execute(queries["Req_tbl"], {"model": modelName}).fetchall()
    # labourline_tbl = server_conn_str.execute(queries["Labour_tbl"], {"model": modelName}).fetchall()
    # partslines_tbl = server_conn_str.execute(queries["Parts_tbl"], {"model": modelName}).fetchall()

    RO_tbl = db_request(queries["RO_tbl"], server_conn_str)
    request_tbl = db_request(queries["Req_tbl"], server_conn_str)
    labourline_tbl = db_request(queries["Labour_tbl"], server_conn_str)
    partslines_tbl = db_request(queries["Parts_tbl"], server_conn_str)

    return RO_tbl, request_tbl, labourline_tbl, partslines_tbl


In [7]:

# # pull data for 
# RO_tbl_vw_174 = db_request(RO_all, vw_18_db)
# request_tbl_vw_174 = db_request(query_all_requests, vw_18_db)
# labourline_tbl_vw_174 = db_request(query_all_LabourLine, vw_18_db)
# partslines_tbl_vw_174 = db_request(query_all_PartsLine, vw_18_db)


In [8]:

# function to drop empty columns
def drop_empty_columns(df):
    df_cleaning = df.copy()
    # drop empty columns - must all empty
    df_cleaning = df_cleaning.dropna(axis=1, how='all')
    
    return df_cleaning
 

In [9]:
# function to filter columns
def filter_for_essential_columns(df, essential_cols):
    df_selected = df[essential_cols].copy()
    return df_selected

In [10]:
# Defined essential columns for each table

essential_columns_request_tbl = ['fldId', 'fldWorkItemRef', 'fldSequence', 'fldDescription',
       'fldRequestCodeRef', 'fldRequestCode', 'fldRequestedTime', 'fldOrderNumber',
        'fldLastUpdated']


essential_cols_labourline_tbl = ['fldID', 'fldRequestRef', 'fldOpCodeRef',
       'fldActualHours', 'fldSoldHours', 'fldDescription',
       'fldAddedDate']

essential_cols_partlines_tbl = ['fldID', 'fldRequestRef', 'fldSequence', 'fldPartNumber', 'fldPartDesc',
       'fldRequested', 'fldShipped', 'fldOrderType', 'fldDateAdded']


essential_cols_RO_tbl = ['fldId', 'fldContactRef', 'fldVehicleRef', 'fldDateOpened',
       'fldDateClosed'
       ]
       
  

In [11]:

def clean_datset(df, tbl_type):
    df_dropped_empty_cols = drop_empty_columns(df)

    if tbl_type == "request":
        df_filtered = filter_for_essential_columns(df_dropped_empty_cols, essential_columns_request_tbl)
    
    elif tbl_type == "labourline":
        df_filtered = filter_for_essential_columns(df_dropped_empty_cols, essential_cols_labourline_tbl)

    elif tbl_type == "partslines":
        df_filtered = filter_for_essential_columns(df_dropped_empty_cols, essential_cols_partlines_tbl)

    elif tbl_type == "RO_tbl":
        df_filtered = filter_for_essential_columns(df_dropped_empty_cols, essential_cols_RO_tbl)
        # remove
        
    return df_filtered
 

In [12]:

# request_tbl_vw_174 = clean_datset(request_tbl_vw_174, tbl_type="request")
# labor_tbl_vw_174 = clean_datset(labourline_tbl_vw_174, tbl_type="labourline")
# parts_tbl_vw_174 = clean_datset(partslines_tbl_vw_174, tbl_type="partslines")
# RO_tbl_vw_174 = clean_datset(RO_tbl_vw_174, tbl_type="RO_tbl")

#### Find a list of labour and parts for the following repair jobs 

- water pump 
- Timing belt
- Electrical - exterior lights


In [13]:
def search_columns_for_keyword(df, keyword, column):
    if (column not in df.columns) or column=="":
        raise ValueError(f"Column '{column}' does not exist in the DataFrame.")
    filtered_df = df[df[column].str.contains(keyword, case=False, na=False)]
    return filtered_df

def get_top_ten_opcodes(df):
    top_ten = df["fldRequestCode"].value_counts().head(20)
    return top_ten

def search_request_by_opcode(df, opcode):
    search_result = df[df["fldRequestCode"]== opcode]
    
    return search_result

def search_request_by_list_of_opcodes(df, opcode_list):
    search_result = df[df["fldRequestCode"].isin(opcode_list)]
    
    return search_result



def part_items_metrics(parts_df):

    uniq_item_by_description = set(parts_df['fldPartDesc'].unique())
    metrics = pd.DataFrame(columns = ['partDesc','#UniqParts','#Qty','uniq_partNumbers'])

    for desc in uniq_item_by_description:
        item_count = len(parts_df[parts_df['fldPartDesc'] == desc]) 
        total_units = parts_df[parts_df['fldPartDesc'] == desc]['fldRequested'].sum()
        uniq_partNumbers = parts_df[parts_df['fldPartDesc'] == desc]['fldPartNumber'].unique().tolist()
        new_row = {
                    'partDesc': desc, 
                    '#UniqParts': item_count, 
                    '#Qty': total_units,
                    'uniq_partNumbers': uniq_partNumbers
                    }
        
        # metrics = pd.DataFrame(columns = ['partDesc','#UniqParts','#Qty','uniq_partNumbers'])
        
        new_row_df = pd.DataFrame([new_row]).reindex(columns=metrics.columns)
        metrics = pd.concat([metrics, new_row_df], ignore_index=True)
    return metrics




# def parts_summary(parts_tbl_df, total_req_count):
#     # Count occurrences of each unique part
#     # part_counts = parts_tbl_df['fldPartDesc'].value_counts()

#     part_counts = parts_tbl_df.groupby("fldPartDesc", as_index=False).agg(
#     count = ("fldPartNumber", "count"),
#     PartNum = ("fldPartNumber", lambda x: list(x.unique()))
#     ).sort_values("count", ascending=False)
#     part_counts = part_counts[~part_counts["fldPartDesc"].str.contains('ENV Fee|Core charge', case=False, regex=True)]


#     # Calculate percentage occurrence
#     part_counts["perc_occurence"] = round((part_counts['count'] / total_req_count * 100), 2)
    

#     # Sort for readability
#     metrics = metrics.sort_values(by='#perc_occurence', ascending=False)


#     # display(metrics)
#     return metrics



def parts_summary_v1(parts_tbl_df, total_req_count, similarity_threshold, ignore_words):
    """
    Summarizes parts occurrence and groups similar descriptions based on keyword similarity.
    
    Parameters:
    ----------
    parts_tbl_df : pd.DataFrame
        DataFrame containing part descriptions and request references.
    total_req_count : int
        Total number of requests for percentage calculation.
    similarity_threshold : float, optional (default=0.2)
        Jaccard similarity threshold for grouping descriptions.
    ignore_words : list of str, optional
        Words to ignore when determining similarity and forming combined names.
    
    Returns:
    -------
    pd.DataFrame
        DataFrame with combined part names and % occurrence.
    """
    
    if ignore_words is None:
        ignore_words = []
    
    # Normalize descriptions
    parts_tbl_df.loc[parts_tbl_df["fldPartDesc"].notna(), "fldPartDesc"] = (
        parts_tbl_df.loc[parts_tbl_df["fldPartDesc"].notna(), "fldPartDesc"].str.upper()
    )

    # Calculate initial metrics
    metrics = (
        parts_tbl_df[['fldPartDesc', 'fldRequestRef']]
        .drop_duplicates()
        .groupby('fldPartDesc')
        .size()
        .reset_index(name='Count')
    )

    # Tokenize descriptions and remove ignored words
    metrics['Tokens'] = metrics['fldPartDesc'].apply(
        lambda x: set(word for word in re.split(r'\W+', x) if word and word not in [w.upper() for w in ignore_words])
    )

    # Group similar descriptions
    grouped = []
    visited = set()

    for i, row_i in metrics.iterrows():
        if i in visited:
            continue
        group = [i]
        for j, row_j in metrics.iterrows():
            if j in visited or i == j:
                continue
            # Jaccard similarity
            sim = len(row_i['Tokens'] & row_j['Tokens']) / len(row_i['Tokens'] | row_j['Tokens'])
            if sim >= similarity_threshold:
                group.append(j)
        visited.update(group)
        grouped.append(group)

    # Aggregate groups
    new_rows = []
    for group in grouped:
        part_names = metrics.loc[group, 'fldPartDesc'].tolist()
        counts = metrics.loc[group, 'Count'].sum()
        common_tokens = set.intersection(*metrics.loc[group, 'Tokens']) if len(group) > 1 else metrics.loc[group, 'Tokens'].iloc[0]
        common_name = " ".join(sorted(common_tokens)) if common_tokens else part_names[0]
        new_rows.append({'Part': common_name, 'Count': counts})

    # Create final DataFrame
    final_df = pd.DataFrame(new_rows)
    final_df['%Occurrence'] = (final_df['Count'] / total_req_count) * 100
    final_df['%Occurrence'] = final_df['%Occurrence'].round(2)
    final_df = final_df.sort_values(by='%Occurrence', ascending=False).reset_index(drop=True)
    final_df = final_df[["Part", "%Occurrence"]]
    return final_df



def parts_summary(parts_tbl_df):
    

    parts_tbl_df.loc[parts_tbl_df["fldPartDesc"].notna(), "fldPartDesc"] = (
        parts_tbl_df.loc[parts_tbl_df["fldPartDesc"].notna(), "fldPartDesc"].str.upper()
    )

    parts_tbl_df = parts_tbl_df[~parts_tbl_df["fldPartDesc"].str.contains("ENV FEE|CORE|Fluids", case=False, regex = True)]

    total_req_count = len(parts_tbl_df['fldRequestRef'].unique())

    metrics = (parts_tbl_df[['fldPartDesc', 'fldRequestRef']]
               .drop_duplicates()
               .groupby('fldPartDesc', as_index=False)
                .agg(
                    uniq_fldPartDesc_count= ("fldRequestRef","nunique")
                    )
                )
        

    metrics["freq_perc"] = (metrics["uniq_fldPartDesc_count"]/total_req_count*100).round(2)

    partNumber_uniqueList = (parts_tbl_df.groupby('fldPartDesc',as_index=False)
                                .agg(PartNumbers = ('fldPartNumber', lambda x: list(pd.unique(x))))
                            )
    
    results = metrics.merge(partNumber_uniqueList, on = 'fldPartDesc')
    results = results.sort_values("freq_perc", ascending=False)

    results = results.reset_index(drop=True).set_axis(range(1, len(results) + 1))


    print(f"Sample size: {total_req_count} ROs")
    # print(f"# Unique opcodes: {len(filtered_df["fldRequestCode"].unique())}")
    results.columns =["Part","Count", "frequency_%", "PartNumbers"]

    return results[["Part", "frequency_%", "PartNumbers"]]



def search_parts_and_labour_by_req_id(labor, parts, req_id):

    if "fldRequestRef" not in labor.columns or "fldRequestRef" not in parts.columns:
        raise ValueError("The required column 'fldRequestRef' does not exist in one of the DataFrames.")
    
    labor_result = labor[labor["fldRequestRef"]== req_id]
    parts_result = parts[parts["fldRequestRef"]== req_id]
        
    print(f"Labour items for Request ID {req_id}:")
    display(labor_result)
    print(f"Parts items for Request ID {req_id}:")
    display(part_items_metrics(parts_result))
    

In [14]:

def remove_invalid_opcodes(df: pd.DataFrame) -> pd.DataFrame:
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"Expected a pandas DataFrame, got {type(df)}")

    required_cols = ['fldFlatHours', 'fldTimeAllowed', 'fldCode']

    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise KeyError(f"Missing required columns: {missing_cols}")

    # Apply filter
    return df[(df["fldFlatHours"] > 0) & (df["fldTimeAllowed"] > 0)]
    # return df



def get_valid_op_codes_by_keyword(db_conn, key_word: str) -> list:
    query = f"SELECT * FROM Ops_tblOpCode2 WHERE fldDescription LIKE '%{key_word}%'"
    search_result = db_request(query, db_conn)

    if search_result.empty:
        raise LookupError(f"No matching data for query: {query}")

    filter_results = remove_invalid_opcodes(search_result)

    if filter_results.empty:
        raise LookupError(f"No matching records found for keyword: {key_word}")

    return filter_results



def get_all_op_codes(db_conn) -> pd.DataFrame:
    """
    Retrieves all opcodes from the database.
    """
    query = "SELECT * FROM Ops_tblOpCode2"
    
    search_result = db_request(query, db_conn)

    return search_result


In [15]:

def clean_data(RO_tbl, request_tbl, labourline_tbl, partslines_tbl):
    RO_tbl_cleaned = clean_datset(RO_tbl, tbl_type="RO_tbl")
    request_tbl_cleaned = clean_datset(request_tbl, tbl_type="request")
    labourline_tbl_cleaned = clean_datset(labourline_tbl, tbl_type="labourline")
    partslines_tbl_cleaned = clean_datset(partslines_tbl, tbl_type="partslines")

    return RO_tbl_cleaned, request_tbl_cleaned, labourline_tbl_cleaned, partslines_tbl_cleaned


In [16]:

# Function to count the number of times a unique part item appears on a repair job

def parts_analysis(part_items, tracker_count_part_item_once_per_job, parts_summary_df):
    for index, row in part_items.iterrows():
            part_number = row["fldPartNumber"]
            part_desc = row["fldPartDesc"]

            # Count occurrence of each part item used on job             
            if (part_number in parts_summary_df["Part Number"].values) and (part_number not in tracker_count_part_item_once_per_job):
                parts_summary_df.loc[parts_summary_df["Part Number"] == part_number, "Occurrence_count"] += 1
                tracker_count_part_item_once_per_job.add(part_number)
            else:
                new_row = {
                    "Part Number": part_number,
                    "Part Description": part_desc,
                    "Occurrence_count": 1
                }
                parts_summary_df = pd.concat([parts_summary_df, pd.DataFrame([new_row])], ignore_index=True) 
                tracker_count_part_item_once_per_job.add(part_number)
                parts_summary_df.sort_values(by="Occurrence_count", ascending=False, inplace=True)


    return parts_summary_df, tracker_count_part_item_once_per_job
 

In [139]:


def filter_rows_by_keywords(df, column_name, keywords=[[], []], return_print=True):
    """
    Filters rows in a DataFrame where:
    - All keywords in the first sub-array must be present (AND logic).
    - At least one keyword in the second sub-array must be present (OR logic).
    
    Parameters:
    ----------
    df : pd.DataFrame
        The DataFrame to search.
    column_name : str
        The name of the column to search within.
    keywords : list of two lists
        keywords[0] = list of must-have keywords (AND condition)
        keywords[1] = list of optional keywords (at least one required)
    return_counts : bool, optional (default=True)
        If True, returns value counts of the filtered column.
        If False, returns the filtered DataFrame.
    
    Returns:
    -------
    pd.Series or pd.DataFrame
        Value counts of the filtered column or the filtered DataFrame.
    """
    
    must_have = keywords[0]
    optional = keywords[1]
    
    # Build regex for must-have keywords (AND logic using lookaheads)
    must_pattern = "".join(f"(?=.*{re.escape(word)})" for word in must_have)
    
    # Build regex for optional keywords (OR logic using |)
    optional_pattern = "|".join(re.escape(word) for word in optional)
    
    # Combine patterns: must-have AND (optional OR empty if none)
    if optional:
        pattern = f"{must_pattern}(?=.*(?:{optional_pattern}))"
    else:
        pattern = must_pattern
    
    # Apply filter
    mask = df[column_name].str.contains(pattern, case=False, regex=True, na=False)
    filtered_df = df[mask]

    # print(f"Sample size: {len(filtered_df)}")
    # print(f"# Unique opcodes: {len(filtered_df["fldRequestCode"].unique())}")
    key_columns= ['fldRequestCode', 'fldDescription']

    
    return filtered_df[key_columns] if return_print else filtered_df


In [18]:
def labour_items_analysis(labour_items, tracker_count_labour_item_once_par_job, labour_summary_df):
    for index, row in labour_items.iterrows():
            op_code = row["fldOpCodeRef"]
            labour_desc = row["fldDescription"]

            if op_code in labour_summary_df["fldOpCodeRef"].values and op_code not in tracker_count_labour_item_once_par_job:
                labour_summary_df.loc[labour_summary_df["fldOpCodeRef"] == op_code, "Occurrence_count"] += 1
                tracker_count_labour_item_once_par_job.add(op_code)
            else:
                new_row = {
                    "fldOpCodeRef": op_code,
                    "fldDescription": labour_desc,
                    "Occurrence_count": 1
                }
                labour_summary_df = pd.concat([labour_summary_df, pd.DataFrame([new_row])] , ignore_index=True )
                tracker_count_labour_item_once_par_job.add(op_code)
                labour_summary_df.sort_values(by="Occurrence_count", ascending=False, inplace=True)  
    return labour_summary_df, tracker_count_labour_item_once_par_job

In [19]:
def requestsLines_analysis(req_id, part_items, labour_items, filtered_requests_df, request_summary_df):
    
    new_row = {
            "Request ID": req_id,
            "Description": filtered_requests_df[filtered_requests_df["fldId"] == req_id ]["fldDescription"].values[0],
            "fldRequestCode": filtered_requests_df[filtered_requests_df["fldId"] == req_id ]["fldRequestCode"].values[0],
            "#_PartItems": len(part_items),
            "#_LaborItems": len(labour_items)
            }

    request_summary_df = pd.concat([request_summary_df, pd.DataFrame([new_row])], ignore_index=True )
    request_summary_df.sort_values(by="#_PartItems", ascending=False, inplace=True)
    
    return request_summary_df

In [20]:


def run_analysis(labourline_tbl, partslines_tbl, request_tbl):


    filtered_requests_df = request_tbl
    
    # search from labour line and part line where fldRequestRef in filtered_requests_df['fldId']
    filtered_labour_df = labourline_tbl[labourline_tbl["fldRequestRef"].isin(filtered_requests_df['fldId'])]
    filtered_parts_df = partslines_tbl[partslines_tbl["fldRequestRef"].isin(filtered_requests_df['fldId'])]


    parts_summary_df = pd.DataFrame(columns=["Part Number", "Part Description", "Occurrence_count"])
    labour_summary_df = pd.DataFrame(columns=["fldOpCodeRef", "fldDescription", "Occurrence_count"])
    requestLine_summary_df = pd.DataFrame(columns=["Request ID", "Description","fldRequestCode", "#_PartItems", "#_LaborItems"])
 
    for items in filtered_requests_df['fldId'].values:
        req_id = items

        tracker_count_part_item_once_per_job = set()
        tracker_count_labour_item_once_par_job = set()
        

    # print(filtered_requests_df[filtered_requests_df["fldId"] == req_id ]["fldDescription"])

        # search for all parts and labour lines for this req_id
        part_items = filtered_parts_df[filtered_parts_df["fldRequestRef"]== req_id]
        labour_items = filtered_labour_df[filtered_labour_df["fldRequestRef"]==req_id]

        requestLine_summary_df = requestsLines_analysis(req_id, part_items, labour_items, filtered_requests_df, requestLine_summary_df)

        if not part_items.empty:
            parts_summary_df, tracker_count_part_item_once_per_job = parts_analysis(
                                                                                    part_items=part_items, 
                                                                                    tracker_count_part_item_once_per_job = tracker_count_part_item_once_per_job, 
                                                                                    parts_summary_df = parts_summary_df
                                                                                    )


        # if not labour_items.empty:
        #     labour_summary_df, tracker_count_labour_item_once_par_job = labour_items_analysis(
        #                                                                                 labour_items=labour_items, 
        #                                                                                 tracker_count_labour_item_once_par_job=tracker_count_labour_item_once_par_job, 
        #                                                                                 labour_summary_df=labour_summary_df
        #                                                                                 ) 
                                                           

    return filtered_parts_df

In [21]:
def plot_stats(requestLine_summary_df):

    parts_stats =  (
    requestLine_summary_df["#_PartItems"]
    .value_counts()
    .reset_index()
    .rename(columns={'index': '#_PartItems', '#_PartItems': '#Parts'})
    )

    labour_stats =  (
        requestLine_summary_df["#_LaborItems"]
        .value_counts()
        .reset_index()
        .rename(columns={'index': '#_LaborItems', '#_LaborItems': '#labour'})
    )

    # Sort for better visualization
    parts_stats = parts_stats.sort_values(by='#Parts').reset_index(drop=True)
    labour_stats = labour_stats.sort_values(by='#labour').reset_index(drop=True)


    # Compute stats for Parts
    mean_parts = parts_stats['#Parts'].mean()
    median_parts = parts_stats['#Parts'].median()
    mode_parts = parts_stats['#Parts'].mode()[0]

    # Compute stats for Labour
    mean_labour = labour_stats['#labour'].mean()
    median_labour = labour_stats['#labour'].median()
    mode_labour = labour_stats['#labour'].mode()[0]


    # Plot Parts line
    plt.plot(parts_stats['#Parts'], parts_stats['count'], color='blue', marker='o', label='Parts')

    # Plot Labour line
    plt.plot(labour_stats['#labour'], labour_stats['count'], color='green', marker='o', label='Labour')


    # Add reference lines for mean
    plt.axvline(mean_parts, color='blue', linestyle='--', alpha=0.5, label=f'Parts Mean: {mean_parts:.2f}')
    plt.axvline(mean_labour, color='green', linestyle='--', alpha=0.5, label=f'Labour Mean: {mean_labour:.2f}')


    # Annotate median and mode
    plt.text(parts_stats['#Parts'].max(), median_parts, f'Median: {median_parts}', color='blue')
    plt.text(parts_stats['#Parts'].max(), mode_parts, f'Mode: {mode_parts}', color='blue')
    plt.text(labour_stats['#labour'].max(), median_labour, f'Median: {median_labour}', color='green')
    plt.text(labour_stats['#labour'].max(), mode_labour, f'Mode: {mode_labour}', color='green')


    # Labels and title
    plt.xlabel('Item Count')
    plt.ylabel('Frequency')
    plt.title('Parts vs Labour Items with Summary Stats')
    plt.legend()
    plt.grid(True)
    plt.show()

In [22]:

import pandas as pd
import re
import ast
from itertools import chain

def _normalize_partnumbers_cell(x):
    if x is None or (isinstance(x, float) and pd.isna(x)) or (isinstance(x, str) and x.strip() == ""):
        return []
    if isinstance(x, (list, tuple, set)):
        return [str(v).strip() for v in x if str(v).strip()]
    if isinstance(x, str):
        s = x.strip()
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set)):
                    return [str(v).strip() for v in parsed if str(v).strip()]
            except Exception:
                pass
        return [p.strip() for p in s.split(",") if p.strip()]
    return [str(x).strip()] if str(x).strip() else []

def _dedupe_preserve_order(items):
    seen = set()
    out = []
    for x in items:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def _glob_to_regex(token: str, match_mode: str = "contains") -> str:
    """
    Convert a keyword with glob wildcards to regex.
      * => .*
      ? => .
    We escape everything else.
    """
    t = str(token).strip()
    if not t:
        return ""

    esc = re.escape(t)
    esc = esc.replace(r"\*", ".*").replace(r"\?", ".")
    if match_mode == "word":
        return rf"\b{esc}\b"
    return esc

def combine_parts_by_keyword_groups(
    df: pd.DataFrame,
    groups: dict,
    part_col: str = "Part",
    freq_col: str = "frequency_%",
    partnums_col: str = "PartNumbers",
    match_mode: str = "contains",  # "contains" or "word"
    dedupe_partnums: bool = True,
    combined_parts_col: str = "CombinedParts",
    matched_keywords_col: str = "MatchedKeywords",
) -> pd.DataFrame:
    """
    groups format:
      {
        "CANONICAL": [[include_any_patterns], [exclude_any_patterns]],
        ...
      }

    include_any_patterns: OR logic (at least one must match)
    exclude_any_patterns: NOT logic (none may match)

    Matching is case-insensitive (case=False) per your requirement.
    """

    required = {part_col, freq_col, partnums_col}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"Missing required columns: {sorted(missing)}")

    if match_mode not in {"contains", "word"}:
        raise ValueError("match_mode must be 'contains' or 'word'")

    work = df.copy()
    work[freq_col] = pd.to_numeric(work[freq_col], errors="coerce").fillna(0)
    work[partnums_col] = work[partnums_col].apply(_normalize_partnumbers_cell)

    part_series = work[part_col].fillna("").astype(str)

    unassigned = pd.Series(True, index=work.index)
    matched_rows_out = []

    for canonical_name, rule in groups.items():
        if not isinstance(rule, (list, tuple)) or len(rule) != 2:
            raise ValueError(f"Group '{canonical_name}' must be [[include_any],[exclude_any]]")

        include_any, exclude_any = rule
        include_any = include_any or []
        exclude_any = exclude_any or []

        if len(include_any) == 0:
            continue

        # Build OR regex for includes
        include_patterns = [_glob_to_regex(k, match_mode) for k in include_any if str(k).strip()]
        include_patterns = [p for p in include_patterns if p]
        include_or = "(?:" + "|".join(include_patterns) + ")"

        include_mask = part_series.str.contains(include_or, case=False, regex=True, na=False)

        # Build OR regex for excludes (if any)
        if exclude_any:
            exclude_patterns = [_glob_to_regex(k, match_mode) for k in exclude_any if str(k).strip()]
            exclude_patterns = [p for p in exclude_patterns if p]
            if exclude_patterns:
                # exclude_or = "(" + "|".join(exclude_patterns) + ")"
                # exclude_mask = part_series.str.contains(exclude_or, case=False, regex=True, na=False)

                
                exclude_or = "(?:" + "|".join(exclude_patterns) + ")"
                exclude_mask = part_series.str.contains(exclude_or, case=False, regex=True, na=False)

            else:
                exclude_mask = pd.Series(False, index=work.index)
        else:
            exclude_mask = pd.Series(False, index=work.index)

        mask = include_mask & (~exclude_mask)

        group_idx = work.index[unassigned & mask]
        if len(group_idx) == 0:
            continue

        group_df = work.loc[group_idx]

        freq_sum = group_df[freq_col].sum()

        all_partnums = list(chain.from_iterable(group_df[partnums_col].tolist()))
        if dedupe_partnums:
            all_partnums = _dedupe_preserve_order(all_partnums)

        combined_parts = _dedupe_preserve_order(group_df[part_col].fillna("").astype(str).tolist())

        matched_rows_out.append({
            part_col: canonical_name,  # <-- group key goes into Part column
            freq_col: float(freq_sum),
            partnums_col: all_partnums,
            combined_parts_col: combined_parts,
            matched_keywords_col: {
                "include_any": include_any,
                "exclude_any": exclude_any
            }
        })

        unassigned.loc[group_idx] = False

    matched_df = pd.DataFrame(
        matched_rows_out,
        columns=[part_col, freq_col, partnums_col, combined_parts_col, matched_keywords_col]
    )

    remaining_df = work.loc[unassigned, [part_col, freq_col, partnums_col]].copy()
    remaining_df[combined_parts_col] = remaining_df[part_col].fillna("").astype(str).apply(lambda x: [x])
    remaining_df[matched_keywords_col] = [{"include_any": [], "exclude_any": []}] * len(remaining_df)

    final_df = pd.concat([matched_df, remaining_df], ignore_index=True)
    final_df[freq_col] = final_df[freq_col].round(2)
    final_df = final_df.sort_values(by=freq_col, ascending=False, kind="mergesort").reset_index(drop=True)

    return final_df


Analysis for Ford Site 130

In [23]:
# def execute_model(request_tbl_df, search_key_words, labour_line_df, parts_line_df, similarity_threshold, ignore_words):
#     requests_filtered= filter_rows_by_keywords(request_tbl_df, "fldDescription", search_key_words, False)

#     # requestLine_summary_df, parts_summary_df, labour_summary_df, filtered_parts_df = run_analysis(labour_line_df, parts_line_df, requests_filtered)
#     filtered_parts_df = run_analysis(labour_line_df, parts_line_df, requests_filtered)

#     # print(f"Parts % occurrence in a {key_wrd_172} job")

#     # display(parts_summary_v1(parts_tbl_df = filtered_parts_df, total_req_count = all_filtered_req_count, similarity_threshold=similarity_threshold, ignore_words=ignore_words))
#     display(parts_summary(parts_tbl_df = filtered_parts_df))


def compare_site_results_side_by_side(df_130, df_172):
    merged_resutls = df_130.merge(df_172, on="Part", how='outer')
    result_with_select_columns = merged_resutls[["Part", "frequency_%_x","frequency_%_y", "PartNumbers_x", "PartNumbers_y"]]

    result_with_select_columns.columns = ["Part", "Freq_130","Freq_172", "PartNumbers_130", "PartNumbers_172"]
    
    result_with_select_columns = result_with_select_columns.sort_values("Freq_130", ascending=False)

    return result_with_select_columns

def execute_model(request_tbl_df, search_key_words, labour_line_df, parts_line_df,
                  similarity_threshold, ignore_words, groups):

    # Normalize input: allow "Water Pump" or [["water","pump"], []]
    if isinstance(search_key_words, str):
        search_key_words = [[search_key_words], []]

    requests_filtered = filter_rows_by_keywords(
        request_tbl_df, "fldDescription", search_key_words, return_print=False
    )

    # Vectorized: filter parts only once
    req_ids = requests_filtered["fldId"].unique()
    filtered_parts_df = parts_line_df[parts_line_df["fldRequestRef"].isin(req_ids)].copy()
    parts_list = parts_summary(filtered_parts_df)

    
    # print("Before combine")
    # display(parts_list)

    print("-----------------------------------------------------------------------------")
    final_df = combine_parts_by_keyword_groups(
        df=parts_list,
        groups=groups,
        part_col="Part",
        freq_col="frequency_%",       # <-- use your actual frequency column name
        partnums_col="PartNumbers",
        match_mode="contains"         # or "word"
    )

    # display(final_df)
    return final_df








In [24]:
# db_server_130 = "DB_server_130"
# # key_wrd = "Water Pump Replace"
# key_wrd_130 = "Water Pump or Gasket - Remove and Install"
# server_conn_db_130 = SERVER_conn(db_server_130)

Illustration for Site 172

In [25]:


# db_server_172 = "DB_server_172"
# # key_wrd = "Water Pump Replace"

# server_conn_db_172 = SERVER_conn(db_server_172)

# RO_tbl_db_172, request_tbl_db_172, labourline_tbl_db_172, partslines_tbl_db_172 = pull_data_by_server(server_conn_db_172)
# RO_tbl_db_172, request_tbl_db_172, labourline_tbl_db_172, partslines_tbl_db_172 = clean_data(RO_tbl_db_172, request_tbl_db_172, labourline_tbl_db_172, partslines_tbl_db_172)



In [26]:


# RO_tbl_db_130, request_tbl_db_130, labourline_tbl_db_130, partslines_tbl_db_130 = pull_data_by_server(server_conn_db_130)
# RO_tbl_db_130, request_tbl_db_130, labourline_tbl_db_130, partslines_tbl_db_130 = clean_data(RO_tbl_db_130, request_tbl_db_130, labourline_tbl_db_130, partslines_tbl_db_130)

 

In [27]:

# # Repair 1: Water pump replace - Site 172

# # search_key_words_water_pump_172 = [["water", "pump"], ["replace", "change", "Replacing", "leak"]]
# search_key_words_water_pump_172 = [["water", "pump"], []]

# print("Water pump - Site 172")
# resutlts_water_pump_site_172 = execute_model(request_tbl_db_172, search_key_words_water_pump_172, labourline_tbl_db_172, partslines_tbl_db_172, similarity_threshold=0.7, ignore_words=[" - ", "kit", "ASY", "Rep"])

# # Repair 1: Water pump replace - Site 130

# # search_key_words_water_pump_130 = [["water", "pump"], ["replace", "change", "Replacing", "leak"]]
# search_key_words_water_pump_130 = [["water", "pump"], []]
# print("Water pump - Site 130")
# resutlts_water_pump_site_130 = execute_model(request_tbl_db_130, search_key_words_water_pump_130, labourline_tbl_db_130, partslines_tbl_db_130, similarity_threshold= 0.6, ignore_words=[" - ", "kit", "ASY", "Rep"])


# # Repair 2 : Catalytic Converter Replace - Site 172

# search_key_words_catalytic_replace_172 = [["Catalytic Converter"], []]
# # filter_rows_by_keywords(request_tbl_db_172, "fldDescription", search_key_words_battery_replace_172, False)

# print("Catalytic Converter - Site 172")
# resutlts_Catalytic_Converter_site_172 = execute_model(request_tbl_df= request_tbl_db_172, search_key_words= search_key_words_catalytic_replace_172, labour_line_df= labourline_tbl_db_172, parts_line_df= partslines_tbl_db_172, similarity_threshold=0.7, ignore_words=[" - "])
 

# # Repair 2 : Catalytic Converter Replace - Site 130

# search_key_words_catalytic_replace_130 = [["Catalytic Converter"], []]
# # filter_rows_by_keywords(request_tbl_db_130, "fldDescription", search_key_words_battery_replace_130, False)

# print("Catalytic Converter - Site 130")
# resutlts_Catalytic_Converter_site_130 = execute_model(request_tbl_df= request_tbl_db_130, search_key_words= search_key_words_catalytic_replace_130, labour_line_df= labourline_tbl_db_130, parts_line_df= partslines_tbl_db_130, similarity_threshold=0.7, ignore_words=[" - "])
 


# # Repair 3 : Power Steering - Site 172

# # search_key_words_catalytic_replace_172 = [["Power steering"], ["replace", "change"]]
# search_key_words_catalytic_replace_172 = [["Power steering"], []]
# print("Power Steering - Site 172")
# resutlts_Power_Steering_site_172 = execute_model(request_tbl_df = request_tbl_db_172, search_key_words = search_key_words_catalytic_replace_172, labour_line_df = labourline_tbl_db_172, parts_line_df = partslines_tbl_db_172, similarity_threshold=0.7, ignore_words=[" - "])


 
# # Repair 3 : Power Steering - Site 130

# # search_key_words_catalytic_replace_130 = [["Power steering"], ["replace", "change"]]
# search_key_words_catalytic_replace_130 = [["Power steering"], []]
# print("Power Steering - Site 130")
# resutlts_Power_Steering_site_130 = execute_model(request_tbl_df = request_tbl_db_130, search_key_words = search_key_words_catalytic_replace_130, labour_line_df = labourline_tbl_db_130, parts_line_df = partslines_tbl_db_130, similarity_threshold=0.7, ignore_words=[" - "]) 
 

In [28]:

# compare_water_pump = resutlts_water_pump_site_172.merge(resutlts_water_pump_site_130, on="Part")
# compare_water_pump


# # print(type(resutlts_water_pump_site_172))

In [29]:
# Construct queries 


def is_validModel(server_conn, model):
    query = f" SELECT * FROM Veh_tblModel WHERE fldName = '{model}' AND fldInActive = 0"

    retults = db_request(query, server_conn)
    return len(retults)

def query_constructor(model):

    Queries= dict()

    RO_all = f"SELECT RO.fldId, RO.fldContactRef, RO.fldVehicleRef, RO.fldDateOpened, RO.fldDateClosed FROM Ops_tblRepairOrder RO WITH(NOLOCK) INNER JOIN Ops_tblRequests REQ WITH(NOLOCK) ON RO.fldId = REQ.fldWorkItemRef INNER JOIN Veh_tblVehicle VEH WITH(NOLOCK) ON VEH.fldId = RO.fldVehicleRef INNER JOIN Veh_tblTrim TR WITH(NOLOCK) ON TR.fldId = VEH.fldTrimRef INNER JOIN Veh_tblModel MOD WITH(NOLOCK) ON MOD.fldId = TR.fldModelRef WHERE 1=1 AND RO.fldStatus = 3 AND RO.fldDivision IN (1) AND MOD.fldName = '{model}' AND REQ.fldAddWorkStatus IN (100, 300, 400)"

    query_all_requests = f"SELECT Req.fldId, Req.fldWorkItemRef, Req.fldSequence, Req.fldDescription, Req.fldRequestCodeRef, Req.fldRequestCode, Req.fldRequestedTime, Req.fldOrderNumber, Req.fldLastUpdated FROM Ops_tblRepairOrder RO WITH(NOLOCK) INNER JOIN Ops_tblRequests REQ WITH(NOLOCK) ON RO.fldId = REQ.fldWorkItemRef INNER JOIN Veh_tblVehicle VEH WITH(NOLOCK) ON VEH.fldId = RO.fldVehicleRef INNER JOIN Veh_tblTrim TR WITH(NOLOCK) ON TR.fldId = VEH.fldTrimRef INNER JOIN Veh_tblModel MOD WITH(NOLOCK) ON MOD.fldId = TR.fldModelRef  WHERE 1=1 AND RO.fldStatus = 3 AND RO.fldDivision IN (1) AND MOD.fldName = '{model}' AND REQ.fldAddWorkStatus IN (100, 300, 400)" 
    query_all_PartsLine = f"SELECT PL.fldID, PL.fldRequestRef, PL.fldSequence, PL.fldPartNumber, PL.fldPartDesc, PL.fldRequested, PL.fldShipped, PL.fldOrderType, PL.fldDateAdded FROM Ops_tblRepairOrder RO WITH(NOLOCK) INNER JOIN Ops_tblRequests REQ WITH(NOLOCK) ON RO.fldId = REQ.fldWorkItemRef INNER JOIN Veh_tblVehicle VEH WITH(NOLOCK) ON VEH.fldId = RO.fldVehicleRef INNER JOIN Veh_tblTrim TR WITH(NOLOCK) ON TR.fldId = VEH.fldTrimRef INNER JOIN Veh_tblModel MOD WITH(NOLOCK) ON MOD.fldId = TR.fldModelRef  INNER JOIN Ops_tblPartsLine PL WITH(NOLOCK) ON PL.fldRequestRef = REQ.fldId WHERE 1=1 AND RO.fldStatus = 3 AND RO.fldDivision IN (1) AND MOD.fldName = '{model}' AND REQ.fldAddWorkStatus IN (100, 300, 400)"

    query_all_LabourLine = f"SELECT LL.fldID, LL.fldRequestRef, LL.fldOpCodeRef, LL.fldActualHours, LL.fldSoldHours, LL.fldDescription, LL.fldAddedDate FROM Ops_tblRepairOrder RO WITH(NOLOCK) INNER JOIN Ops_tblRequests REQ WITH(NOLOCK) ON RO.fldId = REQ.fldWorkItemRef INNER JOIN Veh_tblVehicle VEH WITH(NOLOCK) ON VEH.fldId = RO.fldVehicleRef INNER JOIN Veh_tblTrim TR WITH(NOLOCK) ON TR.fldId = VEH.fldTrimRef INNER JOIN Veh_tblModel MOD WITH(NOLOCK) ON MOD.fldId = TR.fldModelRef INNER JOIN Ops_tblLabourLine LL WITH(NOLOCK) ON LL.fldRequestRef = REQ.fldId WHERE 1=1 AND RO.fldStatus = 3 AND RO.fldDivision IN (1) AND MOD.fldName = '{model}' AND REQ.fldAddWorkStatus IN (100, 300, 400)"

    Queries["RO_tbl"] = RO_all
    Queries["Req_tbl"] = query_all_requests
    Queries["Parts_tbl"] = query_all_PartsLine
    Queries["Labour_tbl"] = query_all_LabourLine

    return Queries

In [30]:


def data_pull(modelName,db_server):
    
    # key_wrd = "Replace Water Pump"
    server_conn = SERVER_conn(db_server)
    
    if not is_validModel(server_conn, modelName):
        raise ValueError("Provided Model Name does not exists")
    queries = query_constructor(modelName)
    RO_tbl, request_tbl, labourline_tbl, partslines_tbl = pull_data_by_server_with_args(server_conn, queries, modelName)
    RO_tbl, request_tbl, labourline_tbl, partslines_tbl = clean_data(RO_tbl, request_tbl, labourline_tbl, partslines_tbl)
    return RO_tbl, request_tbl, labourline_tbl, partslines_tbl


def save_data(directoryPath : str, df, siteName):
    # df.to_csv('dat/site')
    return 


def run_model(key_wrd, request_tbl, labourline_tbl, partslines_tbl):
    execute_model(request_tbl_df = request_tbl, search_key_words = key_wrd, labour_line_df = labourline_tbl, parts_line_df = partslines_tbl, similarity_threshold=0.7, ignore_words=[" - "]) 




    

In [31]:
def dataset_info(RO_tbl, req_tbl, labour_tbl, parts_tbl, site):

    print(f"Sample Size: Site {site}")

    print(f"RO_tbl: {len(RO_tbl)}")
    print(f"Req_tbl: {len(req_tbl)}")
    print(f"LabourLines_tbl: {len(labour_tbl)}")
    print(f"PartLines_tbl: {len(parts_tbl)}")

In [32]:
modelName = "F-150"
Servers = ["DB_server_130",'DB_server_172']

RO_tbl_130_f150, request_tbl_130_f150, labourline_tbl_130_f150, partslines_tbl_130_f150 = data_pull(modelName, Servers[0])

Connected successfully. SQL Server version:
Microsoft SQL Server 2022 (RTM-CU22-GDR) (KB5072936) - 16.0.4230.2 (X64) 
	Nov 25 2025 23:31:11 
	Copyright (C) 2022 Microsoft Corporation
	Developer Edition (64-bit) on Windows Server 2022 Standard 10.0 <X64> (Build 20348: )



In [33]:

# WATER PUMP", "PUMP ASY", "PUMP ASY - WA*", "KIT - WATER"

groups= {
    "WATER PUMP (KIT/ASY)": [["WATER", "PUMP"], ["Gasket","Pulley","HOSE","COVER","OIL","CONNECTION","WATER BYP","FUE","TUBE","ADAPTOR","WASHER"]],
    "MC YELLOW COOLANT":[["COOLANT"], []],
    "GASKET - WATER PUMP ":[["GASKET"],[]],
    "POWER STEERING (ALL)": [["POWER STEERING", "P/S", "STEERING PUMP"],[]],
    "CATALYTIC CONVERTER (ALL)": [["CATALYTIC", "CAT CONVERTER", "CONVERTER"],[]]
}

In [34]:
# key_wrd = "Water Pump"
# results_tbl_130_f150 = execute_model(request_tbl_df = request_tbl_130_f150, search_key_words = key_wrd, labour_line_df = labourline_tbl_130_f150, parts_line_df = partslines_tbl_130_f150, similarity_threshold=0.7, ignore_words=[" - "], groups=groups)

In [84]:

def save_data_locally(data, base_path, server_name):
    # Build final directory: base_path/server_name
    final_path = os.path.join(base_path, server_name)
    
    # Create directory if it doesn't exist
    os.makedirs(final_path, exist_ok=True)
    
    # Basic validation
    if not isinstance(data, dict):
        raise ValueError("`data` must be a dict of pandas DataFrames.")
    
    # Save each DataFrame
    for key, df in data.items():
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"Value for key '{key}' is {type(df)}, expected pandas.DataFrame")

        file_path = os.path.join(final_path, f"{key}.csv")
        print(f"Saving {key} -> {file_path}")  # DEBUG print
        df.to_csv(file_path, index=False)

    print(f"Done. Saved {len(data)} file(s) into: {final_path}")


# # ---- TEST IT ----
# data = {
#     "RO_tbl": pd.DataFrame({"a": [1, 2], "b": [3, 4]}),
#     "Req_tbl": pd.DataFrame({"x": [10, 20]}),
#     "Parts_tbl": pd.DataFrame({"part": ["p1", "p2"]}),
# }

# # Use a path you KNOW exists & can write to
# base_path = "./data"   # <-- safer than "/data/" on many systems

# save_data_locally(data, base_path)

In [95]:
# modelName = "Escape"
# Servers = ["DB_server_130",'DB_server_172']

# RO_tbl_130, request_tbl_130, labourline_tbl_130, partslines_tbl_130 = data_pull(modelName, Servers[0])
# RO_tbl_172, request_tbl_172, labourline_tbl_172, partslines_tbl_172 = data_pull(modelName, Servers[1])



# dataset_info(RO_tbl_130, request_tbl_130, labourline_tbl_130, partslines_tbl_130, "130")
# print("---------------------------------------------------------------------")
# dataset_info(RO_tbl_172, request_tbl_172, labourline_tbl_172, partslines_tbl_172, "172")


# data = {"sever_130": {
#                 "RO_tbl": RO_tbl_130,
#                 "Req_tbl": request_tbl_130, 
#                 "Labor_tbl":labourline_tbl_130, 
#                 "Parts_tbl": partslines_tbl_130},
#         "server_172": {
#                 "RO_tbl": RO_tbl_172,
#                 "Req_tbl": request_tbl_172, 
#                 "Labor_tbl":labourline_tbl_172, 
#                 "Parts_tbl": partslines_tbl_172}
# }
base_path = "./data"   # <-- safer than "/data/" on many systems

# for key, data in data.items():
#      save_data_locally(data, base_path, key)



In [122]:

def data_loading_from_local(base_path, server_name):
    # full_path = os.path.join(base_path, server_name)

    RO_file_name = os.path.join(base_path, server_name, "RO_tbl.csv")
    Req_file_name = os.path.join(base_path, server_name, "Req_tbl.csv")
    Labor_file_name = os.path.join(base_path, server_name, "Labor_tbl.csv")
    Parts_file_name = os.path.join(base_path, server_name, "Parts_tbl.csv")

    RO_df = pd.read_csv(RO_file_name)
    Req_df = pd.read_csv(Req_file_name)
    Labor_df = pd.read_csv(Labor_file_name)
    Parts_df = pd.read_csv(Parts_file_name)

    return RO_df, Req_df, Labor_df, Parts_df


In [123]:
# loading data sets
base_path = "./data"

RO_tbl_130, request_tbl_130, labourline_tbl_130, partslines_tbl_130 = data_loading_from_local(base_path, "server_130")
RO_tbl_172, request_tbl_172, labourline_tbl_172, partslines_tbl_172 = data_loading_from_local(base_path, "server_172")

In [133]:
partslines_tbl_172

,fldID,fldRequestRef,fldSequence,fldPartNumber,fldPartDesc,fldRequested,fldShipped,fldOrderType,fldDateAdded
0,E8814D5D-C63C-40CB-8106-D52483782100,4617016E-575B-4D89-89B6-102A042291B8,1,5w20 5qts,5w20 5 qts,1.0,1.0,0,2015-01-28 09:06:30.637
1,64366A1B-2E08-4061-8B05-9613146E60A7,4617016E-575B-4D89-89B6-102A042291B8,2,XO 5W20 BSP,OIL - ENGINE,5.0,5.0,0,2015-01-28 09:06:30.637
2,03174B35-C87F-465E-A5A1-151B20A88DBE,4617016E-575B-4D89-89B6-102A042291B8,3,f fl910,FL 910,1.0,1.0,0,2015-01-28 09:06:30.637
3,822DB4BB-ED21-461E-90E6-5132ECBFB277,4617016E-575B-4D89-89B6-102A042291B8,4,BE8Z 6731 AC,KIT - ELEMENT & GASKET - OIL F,1.0,1.0,0,2015-01-28 09:06:30.637
4,322D9029-9605-4413-8C3D-05603826C961,D6B7093B-A34F-48FE-A1A8-0DB4D26A8979,1,5w30 6qts,5w30 6qts,1.0,1.0,0,2016-11-28 07:06:09.917
...,...,...,...,...,...,...,...,...,...
177914,016AFCBE-954B-410C-B70F-0E417BDA3811,215CC85D-8A33-439C-A3C5-FFDD051AFF51,1,NPN,NPN Part UCS History,1.0,1.0,0,2012-02-24 00:00:00.000
177915,96B7680D-4F33-4BCB-A283-3BDE974AAB95,9A9A017A-7FED-40BF-811C-FFEEBE4E9829,1,NPN,NPN Part UCS History,1.0,1.0,0,2008-12-18 00:00:00.000
177916,7D94E382-0CCE-4F12-98E7-8B0F0D6AA0D4,B640D1F7-08B3-489C-9722-FFFDAAC73B18,1,NPN,NPN Part UCS History,1.0,1.0,0,2009-10-21 00:00:00.000
177917,ECC04D33-988A-4B40-B659-7098417C78CE,5A0A0661-A300-44FF-A471-FF1D3C7011E7,1,NPN,non oem part **aftermarket fender**,1.0,1.0,0,2013-01-29 08:02:46.063


In [ ]:

key_wrds = ["Water Pump", "Power steering", "Catalytic Converter"]

key_wrd = key_wrds[0]

results130_water_pump = execute_model(request_tbl_df = request_tbl_130, search_key_words = key_wrd, labour_line_df = labourline_tbl_130, parts_line_df = partslines_tbl_130, similarity_threshold = 0.7, ignore_words=[" - "], groups=groups)
results172_water_pump = execute_model(request_tbl_df = request_tbl_172, search_key_words = key_wrd, labour_line_df = labourline_tbl_172, parts_line_df = partslines_tbl_172, similarity_threshold = 0.7, ignore_words=[" - "], groups=groups)



Sample size: 139 ROs
-----------------------------------------------------------------------------
Sample size: 77 ROs
-----------------------------------------------------------------------------


In [142]:
temp = partslines_tbl_130[partslines_tbl_130["fldPartDesc"].str.contains("pump", case=False, na=False)]
temp["fldPartDesc"].value_counts()

fldPartDesc
GASKET - WATER PUMP          168
PUMP ASY - WAT               160
PUMP ASY - WATER             134
PUMP ASY - VACUUM             82
GASKET - VACUUM PUMP          62
PUMP ASY - OIL                53
PUMP ASY - FUE                11
PUMP ASY - VAC                11
SENDER AND PUMP ASY           11
MOTOR AND PUMP                10
PUMP ASY - FUEL               10
MOTOR AND PUMP ASY             9
PUMP ASY - POW                 8
PUMP ASY                       7
PUMP ASSY (21S38)  SSSC        3
PULLEY - WATER PUMP            2
VACUUM PUMP GASKET             2
PUMP ASSY (SSSC)               2
FUEL PUMP                      2
FUEL PUMP A/M                  1
WATER PUMP                     1
WATER PUMP BEL                 1
PUMP ASSY                      1
PUMPASY-OIL                    1
FUEL PUMP AND SENDER ASSY      1
FUEL PUMP ASSY                 1
STEERING PUMP                  1
PUMP ASSY (21S38)              1
COUPLING - PUMP DRIV           1
PIPE - FUEL PUMP FEE           

In [143]:
compare_site_results_side_by_side(results130_water_pump, results172_water_pump)

,Part,Freq_130,Freq_172,PartNumbers_130,PartNumbers_172
81,WATER PUMP (KIT/ASY),95.69,100.00,"[PW 556, PW 545, PW 625, PW 686, PW 579, PW 49...","[DS7Z 8501 E, 7S7Z 8501 C, 4S4Z 8501 AA, 4S4Z ..."
24,GASKET - WATER PUMP,79.14,85.72,"[BE8Z 8507 A, 1S7Z 8507 AE, 9L8Z 8507 A, CV6Z ...","[BE8Z 8507 A, BM5Z 6584 B]"
37,MC YELLOW COOLANT,46.76,1.30,"[CVC 13 DLG, CVC 13 G, CVC 7 B2, CVC 7 D1LB2]",[nan]
41,NUT,33.09,6.49,"[W520415 S442, W520214 S440, W715135 S440]","[W715618 S437, W520214 S440, W520415 S442, W52..."
4,ANTI-FREEZE,30.94,24.68,"[CVC 7 B2, CVC 13 DLG, CVC 13 G, CVC 3 B2, CVC...","[VC 13 G, VC 3 B, VC 7 B]"
...,...,...,...,...,...
57,RETAINER - NUT,NaN,2.60,NaN,[CCPZ 3B477 G]
68,SEPARATOR ASY - OIL,NaN,1.30,NaN,[DS7Z 6A785 C]
70,SHAFT - FRONT AXLE,NaN,1.30,NaN,[CV6Z 3B436 B]
76,TENSIONER - TIMING BELT,NaN,1.30,NaN,[BM5Z 6K254 A]


In [144]:
key_wrd = key_wrds[1]
results130_power_str = execute_model(request_tbl_df = request_tbl_130, search_key_words = key_wrd, labour_line_df = labourline_tbl_130, parts_line_df = partslines_tbl_130, similarity_threshold=0.7, ignore_words=[" - "], groups=groups)
results172_power_str = execute_model(request_tbl_df = request_tbl_172, search_key_words = key_wrd, labour_line_df = labourline_tbl_172, parts_line_df = partslines_tbl_172, similarity_threshold=0.7, ignore_words=[" - "], groups=groups)

Sample size: 68 ROs
-----------------------------------------------------------------------------
Sample size: 129 ROs
-----------------------------------------------------------------------------


In [ ]:

compare_site_results_side_by_side(results130_power_str, results172_power_str)
 

,Part,Freq_130,Freq_172,PartNumbers_130,PartNumbers_172
56,SENSOR - STEER,44.12,NaN,[CL8Z 3F818 A],NaN
6,BOLT,39.71,33.33,"[W712250 S437, W713065 S439]","[W714807 S900, W716075 S442, 7N5Z 00812 A, W71..."
10,BULK MERCON V,10.29,NaN,[CXT-5-L],NaN
66,WATER PUMP (KIT/ASY),8.82,NaN,"[STP 152, STP 182, ZLCRD21-5271]",NaN
21,GEAR ASY - STE,5.88,NaN,"[STE 98, STE 419, STE 282, STE 175]",NaN
...,...,...,...,...,...
58,SENSOR - STEERING ROTATION,NaN,38.76,NaN,[CL8Z 3F818 A]
59,SENSOR ASY,NaN,2.33,NaN,"[HV6Z 2C204 A, LX6Z 2C190 A]"
61,SHOCK ABSORBER ASY - FRONT,NaN,0.78,NaN,[CV6Z 18124 AZ]
62,SPRING - FRONT,NaN,0.78,NaN,[GJ5Z 5310 G]


In [146]:
key_wrd = key_wrds[2]
results130_catal = execute_model(request_tbl_df = request_tbl_130, search_key_words = key_wrd, labour_line_df = labourline_tbl_130, parts_line_df = partslines_tbl_130, similarity_threshold=0.7, ignore_words=[" - "], groups=groups)
results172_catal = execute_model(request_tbl_df = request_tbl_172, search_key_words = key_wrd, labour_line_df = labourline_tbl_172, parts_line_df = partslines_tbl_172, similarity_threshold=0.7, ignore_words=[" - "], groups=groups)

Sample size: 11 ROs
-----------------------------------------------------------------------------
Sample size: 0 ROs
-----------------------------------------------------------------------------


In [147]:
compare_site_results_side_by_side(results130_catal, results172_catal)

,Part,Freq_130,Freq_172,PartNumbers_130,PartNumbers_172
5,CATALYTIC CONVERTER (ALL),81.82,NaN,"[LX6Z 5E212 KZ, JJ5Z 5E212 B, CV6Z 5E212 D, CV...",NaN
0,BOLT,63.64,NaN,"[W500233 S442, W715681 S900, W716075 S442, W71...",NaN
14,NUT - HEX.,63.64,NaN,"[W520103 S442, W520103 S403, W520203 S442]",NaN
8,CLAMP - EXHAUST,45.45,NaN,"[JX6Z 5A215 C, LX6Z 5A215 D, LX6Z 5A215 A]",NaN
9,GASKET - WATER PUMP,45.45,NaN,"[ZL31578, AM5Z 9450 A, CV6Z 9450 E, BB5Z 6L612...",NaN
3,BOLT AND WASHER ASY,18.18,NaN,"[W709601 S442, W711806 S442]",NaN
13,NUT,18.18,NaN,"[W716271 S437, W520415 S442]",NaN
22,SEAL - REFER TO (PK-CN1Z) **,18.18,NaN,[CN1Z 7H424 B],NaN
21,SEAL,18.18,NaN,[CV6Z 7086 B],NaN
19,RETAINER - BEARING,18.18,NaN,[YS4Z 3N324 AA],NaN


In [148]:
# looking at the different unique variations of key parts eg. "Water pump" kit names to determine the best way to filter the data for accuracy.
water_pump_keywrds = ["Water|Pump", "Gasket|Pulley|HOSE|COVER|OIL|CONNECTION|WATER BYP|FUE|TUBE|ADAPTOR|WASHER"] 


In [149]:

all_parts_dfs_combined = results172_catal.copy()

all_parts_dfs_combined = pd.concat(
    [partslines_tbl_172, partslines_tbl_130])


In [150]:
def group_dfs(*dfs):
    return pd.concat(dfs, ignore_index=True)


In [151]:

def search_parts_by_keyword(df, search__keys):
    result = df[(df["fldPartDesc"].str.contains(search__keys[0], case= False, na=False)) &
                   (~df["fldPartDesc"].str.contains(search__keys[1], case= False, na=False))]
    display(result["fldPartDesc"].unique().tolist())
 

In [152]:
results130_catal

,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
0,CATALYTIC CONVERTER (ALL),81.82,"[LX6Z 5E212 KZ, JJ5Z 5E212 B, CV6Z 5E212 D, CV...","[CONVERTER ASY, REAR CONVERTER]","{'include_any': ['CATALYTIC', 'CAT CONVERTER',..."
1,BOLT,63.64,"[W500233 S442, W715681 S900, W716075 S442, W71...",[BOLT],"{'include_any': [], 'exclude_any': []}"
2,NUT - HEX.,63.64,"[W520103 S442, W520103 S403, W520203 S442]",[NUT - HEX.],"{'include_any': [], 'exclude_any': []}"
3,GASKET - WATER PUMP,45.45,"[ZL31578, AM5Z 9450 A, CV6Z 9450 E, BB5Z 6L612...","[GASKET, GASKET - EXHAUST MAN]","{'include_any': ['GASKET'], 'exclude_any': []}"
4,CLAMP - EXHAUST,45.45,"[JX6Z 5A215 C, LX6Z 5A215 D, LX6Z 5A215 A]",[CLAMP - EXHAUST],"{'include_any': [], 'exclude_any': []}"
5,BOLT AND WASHER ASY,18.18,"[W709601 S442, W711806 S442]",[BOLT AND WASHER ASY],"{'include_any': [], 'exclude_any': []}"
6,NUT,18.18,"[W716271 S437, W520415 S442]",[NUT],"{'include_any': [], 'exclude_any': []}"
7,SEAL - REFER TO (PK-CN1Z) **,18.18,[CN1Z 7H424 B],[SEAL - REFER TO (PK-CN1Z) **],"{'include_any': [], 'exclude_any': []}"
8,SEAL,18.18,[CV6Z 7086 B],[SEAL],"{'include_any': [], 'exclude_any': []}"
9,RETAINER - BEARING,18.18,[YS4Z 3N324 AA],[RETAINER - BEARING],"{'include_any': [], 'exclude_any': []}"


In [153]:
# "gasket", "coolant", "Nut", "Anti|Freeze",
# key_words_for_water_pump_parts = ["belt", "nut", "bolt", "screw", "retainer", "seal", "oil", "stud"]

key_words_for_catal_parts = ["converter", "bolt", 'nut', "gasket", "clamp", "washer", "seal", "retainer", "coolant", "hanger", "tube", "sensor", "insulator"]

for part_key_wrd in key_words_for_catal_parts:
    display(results130_catal[results130_catal["Part"].str.contains(part_key_wrd, case=False)])

,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
0,CATALYTIC CONVERTER (ALL),81.82,"[LX6Z 5E212 KZ, JJ5Z 5E212 B, CV6Z 5E212 D, CV...","[CONVERTER ASY, REAR CONVERTER]","{'include_any': ['CATALYTIC', 'CAT CONVERTER',..."


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
1,BOLT,63.64,"[W500233 S442, W715681 S900, W716075 S442, W71...",[BOLT],"{'include_any': [], 'exclude_any': []}"
5,BOLT AND WASHER ASY,18.18,"[W709601 S442, W711806 S442]",[BOLT AND WASHER ASY],"{'include_any': [], 'exclude_any': []}"
13,BOLT - FLANGED HEX.,9.09,[W500545 S900],[BOLT - FLANGED HEX.],"{'include_any': [], 'exclude_any': []}"
26,BOLT (BAG INCL QTY 3X),9.09,[LX6Z 4B496 A],[BOLT (BAG INCL QTY 3X)],"{'include_any': [], 'exclude_any': []}"


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
2,NUT - HEX.,63.64,"[W520103 S442, W520103 S403, W520203 S442]",[NUT - HEX.],"{'include_any': [], 'exclude_any': []}"
6,NUT,18.18,"[W716271 S437, W520415 S442]",[NUT],"{'include_any': [], 'exclude_any': []}"
10,NUT - LOCKING,18.18,[W520102 S442],[NUT - LOCKING],"{'include_any': [], 'exclude_any': []}"
24,NUT - SPECIAL,9.09,[W709729 S442],[NUT - SPECIAL],"{'include_any': [], 'exclude_any': []}"


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
3,GASKET - WATER PUMP,45.45,"[ZL31578, AM5Z 9450 A, CV6Z 9450 E, BB5Z 6L612...","[GASKET, GASKET - EXHAUST MAN]","{'include_any': ['GASKET'], 'exclude_any': []}"


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
4,CLAMP - EXHAUST,45.45,"[JX6Z 5A215 C, LX6Z 5A215 D, LX6Z 5A215 A]",[CLAMP - EXHAUST],"{'include_any': [], 'exclude_any': []}"
18,CLAMP,9.09,[ZL33212],[CLAMP],"{'include_any': [], 'exclude_any': []}"
19,CLAMP - EXHAUS,9.09,[CV6Z 5A231 C],[CLAMP - EXHAUS],"{'include_any': [], 'exclude_any': []}"


...

,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
11,MC YELLOW COOLANT,9.09,[CVC 13 DLG],[MC YELLOW COOLANT 4L (PREMIX)],"{'include_any': ['COOLANT'], 'exclude_any': []}"


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
12,HANGER,9.09,[ZLAPE8531],[HANGER],"{'include_any': [], 'exclude_any': []}"


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
14,TUBE ASY,9.09,[JX6Z 6758 E],[TUBE ASY],"{'include_any': [], 'exclude_any': []}"


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
16,SENSOR - EXHAUST GAS,9.09,[DY 1377],[SENSOR - EXHAUST GAS],"{'include_any': [], 'exclude_any': []}"
17,SENSOR - EXHAU,9.09,[DY 1189],[SENSOR - EXHAU],"{'include_any': [], 'exclude_any': []}"


,Part,frequency_%,PartNumbers,CombinedParts,MatchedKeywords
22,INSULATOR - FLOOR,9.09,[CV6Z 11130 A],[INSULATOR - FLOOR],"{'include_any': [], 'exclude_any': []}"


In [154]:
# looking at the different unique variations of key parts eg. "Water pump" kit names to determine the best way to filter the data for accuracy.
# water_pump_keywrds = ["Water Pump|Pump", "Gasket|Pulley|HOSE|COVER|OIL|CONNECTION|WATER BYP|FUE|TUBE|ADAPTOR|WASHER|Coupling|Steering|coolant"] 

water_pump_gasket_keywrds = ["gasket - water", "None"] 

# water_pump_keywrds
combo_df = group_dfs(partslines_tbl_130, partslines_tbl_172)

search_parts_by_keyword(combo_df, water_pump_gasket_keywrds)


['GASKET - WATER', 'GASKET - WATER PUMP']

In [155]:
# looking at the different unique variations of key parts eg. "Water pump" kit names to determine the best way to filter the data for accuracy.
power_steering_keywrds = ["steering", "pump"] 

search_parts_by_keyword(combo_df, power_steering_keywrds)


['WHEEL ASY - STEERING',
 'SENSOR - STEERING RO',
 'STEERING TORQUE SENSOR',
 'LOCK ASY - STEERING',
 'GEAR ASY - STEERING',
 'used steering column',
 'COLUMN ASY - STEERING',
 'SWITCH ASY - STEERING WHEEL',
 'CORE - GEAR ASY - STEERING',
 'P&A STEERING SHAFT BOLT',
 'SENSOR ASY - STEERING ROTATION',
 'SENSOR - STEERING ROTATION',
 'LOCK ASY - STEERING AND IGNITI',
 'LOCK ASY - STEERING ']